# Basic 01 - Load and Inspect
Small, linear workflow: load -> summary -> contacts/publications -> pick study/assay/run -> load data with metadata.

Run the import and dataset-loading cells first. Each following example selects what it needs independently; change its study, assay, or run to explore your data.


## Imports and dataset path


In [ ]:
from pathlib import Path

from isa_phm import ISAWrapper

import pandas as pd
from IPython.display import display

# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## Load the wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
)


## Summary


In [ ]:
display(wrapper.summary())


## Contacts


In [ ]:
display(wrapper.contacts())


## Publications


In [ ]:
display(wrapper.publications())


## Pick first study


In [ ]:
study = wrapper.study(1)  # Or select by study name
study.title


## Pick first assay


In [ ]:
study = wrapper.study(1)

assay = study.assay(1)  # Or select by assay ID or sensor name
assay.assay_id


## Pick first run


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)

run = assay.run(1)  # Or select by run ID
run.run_id


## Load one run with resolved file metadata


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)

df, meta = run.load_dataframe_with_meta(file_type="auto")


## Show load metadata


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)
df, meta = run.load_dataframe_with_meta(file_type="auto")
display(pd.DataFrame([meta.model_dump()]))


## Show first rows


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)
df, meta = run.load_dataframe_with_meta(file_type="auto")

display(df.head(10))


## Optional: AI export


In [ ]:
ctx = wrapper.ai_context(include_validation=False)
list(ctx.keys())


## List AI tools


In [ ]:
tools_df = pd.DataFrame(wrapper.list_tools())
display(tools_df)


## Call tool: metadata context


In [ ]:
tool_ctx = wrapper.call_tool("ai_context", {
    "include_validation": False,
})
tool_ctx["ok"], sorted(tool_ctx["result"].keys())


## Call tool: run data + load metadata (JSON-safe)


In [ ]:
study = wrapper.study(1)
assay = study.assay(1)
run = assay.run(1)

tool_df = wrapper.call_tool("load_dataframe_with_meta", {
    "study_id": study.title,
    "assay_id": assay.assay_id,
    "run_id": run.run_id,
    "file_type": "auto",
    "head_rows": 5,
})
display(pd.DataFrame([tool_df["result"]["metadata"]]))
display(pd.DataFrame(tool_df["result"]["dataframe"]["head"]))


## Extended metadata tables

Use `wrapper.extensive_summary()` when you want all metadata tables at once.


In [ ]:
tables = wrapper.extensive_summary()
for name, df in tables.items():
    print(f"{name}: {df.shape}")


## Clear DataFrame cache

`clear_cache()` evicts all cached DataFrames from the wrapper integrator.


In [ ]:
wrapper.clear_cache()
print("DataFrame cache cleared.")
